# 04 · Figuras do README

Gera as 4 figuras que contam a história do projeto no README, com títulos em forma de mensagem e os números-chave anotados. As análises completas estão no `02_eda`.

**Entrada:** `workspace.tmw_loyalty.abt_churn` (só safras de treino, `flOOT = 0`) · **Saída:** `output/figs/readme/`

| # | Figura | Mensagem |
| --- | --- | --- |
| 1 | `01_problema` | Quase metade dos ativos sai em 28 dias |
| 2 | `02_recencia` | Uma semana sem aparecer muda tudo |
| 3 | `03_matriz_rf` | Recência e frequência juntas separam o risco de 1% a 80% |
| 4 | `04_lives` | Quem acompanha as lives quase não sai |
| + | `05_tendencia` | Hipótese refutada: tendência não explica o churn (opcional) |

In [0]:
%run ./00_setup

In [0]:
base = spark.table(f'{SCHEMA}.abt_churn').filter('flOOT = 0').toPandas()
base['tendencia'] = pd.Categorical(base['tendencia'], ['caindo', 'estável', 'crescendo'], ordered=True)

README_DIR = f'{FIG_DIR}/readme'
os.makedirs(README_DIR, exist_ok=True)
MEDIA = base['churn'].mean()

print(f'{len(base):,} observações | {base.IdCliente.nunique():,} usuários | '
      f'{base.dtRef.nunique()} safras | churn médio {MEDIA:.1%}'.replace(',', '.'))

## Funções de apoio

Padrão visual do README: título com a mensagem, subtítulo com o contexto, fonte no rodapé e números-chave escritos no gráfico.

In [0]:
FONTE = 'Fonte: sistema de pontos Téo Me Why · usuários ativos, safras mensais de mar/24 a jul/26'

def moldura(fig, ax, titulo, subtitulo):
    """Título (mensagem), subtítulo (contexto) e fonte no rodapé."""
    ax.set_title('')
    fig.suptitle(titulo, x=0.01, ha='left', fontsize=15, fontweight='bold', color=COLOR_TITLE)
    fig.text(0.01, 0.905, subtitulo, ha='left', fontsize=10, color=COLOR_DARK)
    fig.text(0.01, -0.06, FONTE, ha='left', fontsize=8, color=COR_NEUTRA)
    fig.subplots_adjust(top=0.82)


def salvar_readme(fig, arquivo):
    fig.savefig(f'{README_DIR}/{arquivo}.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.show()
    plt.close(fig)


def pct(x):
    return f'{x:.0%}'

## 1. O problema: quase metade dos ativos sai em 28 dias

In [0]:
por_safra = base.groupby('dtRef')['churn'].mean().sort_index()
x = pd.to_datetime(por_safra.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, por_safra * 100, marker='o', color=COLOR_BLUE, lw=2)
ax.fill_between(x, por_safra * 100, color=COLOR_BLUE, alpha=0.12)
ax.axhline(MEDIA * 100, ls='--', color=COR_NEUTRA)
ax.text(x[0], MEDIA * 100 + 1.5, f'média {pct(MEDIA)}', color=COR_NEUTRA, fontsize=10)

for fn in (por_safra.idxmin, por_safra.idxmax):          # anota mínimo e máximo
    d = fn()
    ax.annotate(pct(por_safra[d]), (pd.Timestamp(d), por_safra[d] * 100), textcoords='offset points',
                xytext=(0, 10), ha='center', fontweight='bold', color=COLOR_TITLE)

ax.set(ylim=(0, 100), ylabel='taxa de churn (%)', xlabel='')
ax.grid(axis='x', visible=False)
moldura(fig, ax, f'Quase metade dos usuários ativos sai em 28 dias ({pct(MEDIA)})',
        f'Taxa de churn por safra: varia de {pct(por_safra.min())} a {pct(por_safra.max())}, sem tendência de alta ou queda')
salvar_readme(fig, '01_problema')

## 2. O sinal mais forte de volume: uma semana sem aparecer muda tudo

In [0]:
rec = base.groupby('recencia')['churn'].agg(taxa='mean', n='size')
rec = rec[rec['n'] >= 30]                                 # só dias com amostra suficiente

fig, ax = plt.subplots(figsize=(12, 5))
ax.axvspan(7.5, rec.index.max() + 0.5, color=COLOR_TITLE, alpha=0.06)
ax.plot(rec.index, rec['taxa'] * 100, marker='o', color=COLOR_BLUE, lw=2)
ax.axhline(MEDIA * 100, ls='--', color=COR_NEUTRA)
ax.text(rec.index.max(), MEDIA * 100 - 5, f'média {pct(MEDIA)}', color=COR_NEUTRA, ha='right')

destaques = {1: 'apareceu ontem', 6: '6º dia'}
for dia, texto in destaques.items():
    if dia in rec.index:
        t = rec.loc[dia, 'taxa']
        ax.annotate(f'{pct(t)}\n{texto}', (dia, t * 100), textcoords='offset points', xytext=(-6, 18),
                    fontweight='bold', color=COLOR_TITLE)
depois = rec.loc[rec.index >= 16, 'taxa']
ax.text(16, min(depois.max() * 100 + 6, 94), f'depois de 16 dias: {pct(depois.min())} a {pct(depois.max())}',
        fontweight='bold', color=COLOR_TITLE)

ax.set(ylim=(0, 100), xlim=(0.5, rec.index.max() + 0.5), xticks=range(1, rec.index.max() + 1, 3),
       xlabel='dias desde a última transação', ylabel='taxa de churn (%)')
moldura(fig, ax, 'Uma semana sem aparecer muda tudo',
        'Taxa de churn por dias desde a última transação · área sombreada = mais de 7 dias sem aparecer')
salvar_readme(fig, '02_recencia')

## 3. Recência e frequência juntas separam o risco

In [0]:
base['faixaRecencia'] = pd.cut(base['recencia'], [0, 3, 7, 14, 21, 28],
                               labels=['1-3', '4-7', '8-14', '15-21', '22-28'])
base['faixaFrequencia'] = pd.cut(base['diasAtivosD28'], [0, 1, 2, 4, 8, 28],
                                 labels=['1', '2', '3-4', '5-8', '9+'])

taxa = base.pivot_table(index='faixaFrequencia', columns='faixaRecencia', values='churn',
                        aggfunc='mean', observed=True) * 100
n = base.pivot_table(index='faixaFrequencia', columns='faixaRecencia', values='churn',
                     aggfunc='size', observed=True)
taxa = taxa.where(n >= 30).iloc[::-1]                     # mais ativos no topo

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(taxa, annot=True, fmt='.0f', cmap=CMAP_MARCA, vmin=0, vmax=100, linewidths=1, linecolor='white',
            annot_kws={'fontsize': 12, 'fontweight': 'bold'}, cbar_kws={'label': 'taxa de churn (%)'}, ax=ax)
ax.grid(False)
ax.set(xlabel='dias desde a última transação', ylabel='dias ativos nos últimos 28')
ax.tick_params(axis='y', labelrotation=0)
moldura(fig, ax, f'Recência e frequência juntas: de {taxa.min().min():.0f}% a {taxa.max().max():.0f}% de churn',
        'Taxa de churn (%) por combinação · células com menos de 30 observações ficam em branco')
salvar_readme(fig, '03_matriz_rf')

## 4. O que é único deste negócio: presença nas lives

In [0]:
faixas = ['0-25%', '25-50%', '50-75%', '75-100%']
base['faixaLives'] = pd.cut(base['pctLivesPresente'], [0, 0.25, 0.5, 0.75, np.inf], labels=faixas, include_lowest=True)
res = _resumo(base, 'faixaLives').reindex(faixas)

fig, ax = plt.subplots(figsize=(10, 5))
cores = [COLOR_TITLE if t > MEDIA else COLOR_BLUE for t in res['taxa']]
barras = ax.bar(faixas, res['taxa'] * 100, color=cores, width=0.6)
ax.bar_label(barras, labels=[f'{t:.1%}'.replace('.', ',') for t in res['taxa']],
             padding=4, fontsize=12, fontweight='bold', color=COLOR_TITLE)
ax.axhline(MEDIA * 100, ls='--', color=COR_NEUTRA)
ax.text(3.45, MEDIA * 100 - 5, f'média {pct(MEDIA)}', color=COR_NEUTRA, ha='right', fontsize=9)
ax.set_xticks(range(len(faixas)), [f'{f}\n(n={n:,})'.replace(',', '.') for f, n in zip(faixas, res['n'])])
ax.set(ylim=(0, 100), xlabel='% dos dias de live com presença (últimos 28 dias)', ylabel='taxa de churn (%)')
ax.grid(axis='x', visible=False)
moldura(fig, ax, 'Quem acompanha as lives quase não sai',
        f"Churn cai de {res['taxa'].iloc[0]:.0%} para {res['taxa'].iloc[-1]:.0%} conforme a presença nas lives aumenta")
salvar_readme(fig, '04_lives')

## 5.Hipótese refutada: a tendência de atividade não explica o churn

In [0]:
taxa_t = base.pivot_table(index='faixaFrequencia', columns='tendencia', values='churn',
                          aggfunc='mean', observed=True) * 100
n_t = base.pivot_table(index='faixaFrequencia', columns='tendencia', values='churn',
                       aggfunc='size', observed=True)
taxa_t = taxa_t.where(n_t >= 30).iloc[::-1]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(taxa_t, annot=True, fmt='.0f', cmap=CMAP_MARCA, vmin=0, vmax=100, linewidths=1, linecolor='white',
            annot_kws={'fontsize': 12, 'fontweight': 'bold'}, cbar_kws={'label': 'taxa de churn (%)'}, ax=ax)
ax.grid(False)
ax.set(xlabel='tendência (dias ativos: últimos 28 vs 28 anteriores)', ylabel='dias ativos nos últimos 28')
ax.tick_params(axis='y', labelrotation=0)
moldura(fig, ax, 'Com a mesma frequência, a tendência não muda o churn',
        'Em cada linha as cores quase não mudam: o que parecia tendência era volume')
salvar_readme(fig, '05_tendencia')

## Conferência

Lista as figuras geradas e o trecho pronto para colar no README.

In [0]:
figuras = sorted(f for f in os.listdir(README_DIR) if f.endswith('.png'))
print('\n'.join(figuras))
print()
print('\n'.join(f'![{f[3:-4]}](output/figs/readme/{f})' for f in figuras))